# Graph Analysis Notebook

This notebook provides tools for analyzing the knowledge graph structure, communities, and centrality metrics.

In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded successfully!')

## 1. Load Knowledge Graph

In [ ]:
from backend.core.knowledge_graph import GraphBuilder, CommunityDetector

# Initialize graph builder
graph_builder = GraphBuilder()

# Create sample graph for demonstration
sample_entities = [
    {'text': 'Machine Learning', 'entity_type': 'CONCEPT'},
    {'text': 'Deep Learning', 'entity_type': 'CONCEPT'},
    {'text': 'Neural Networks', 'entity_type': 'CONCEPT'},
    {'text': 'Natural Language Processing', 'entity_type': 'CONCEPT'},
    {'text': 'Computer Vision', 'entity_type': 'CONCEPT'},
    {'text': 'Transformer', 'entity_type': 'CONCEPT'},
    {'text': 'BERT', 'entity_type': 'CONCEPT'},
    {'text': 'GPT', 'entity_type': 'CONCEPT'},
    {'text': 'Geoffrey Hinton', 'entity_type': 'PERSON'},
    {'text': 'Yann LeCun', 'entity_type': 'PERSON'},
]

sample_relations = [
    {'source_text': 'Deep Learning', 'target_text': 'Machine Learning', 'relation_type': 'IS_A'},
    {'source_text': 'Neural Networks', 'target_text': 'Deep Learning', 'relation_type': 'PART_OF'},
    {'source_text': 'Natural Language Processing', 'target_text': 'Machine Learning', 'relation_type': 'IS_A'},
    {'source_text': 'Computer Vision', 'target_text': 'Machine Learning', 'relation_type': 'IS_A'},
    {'source_text': 'Transformer', 'target_text': 'Deep Learning', 'relation_type': 'USES'},
    {'source_text': 'BERT', 'target_text': 'Transformer', 'relation_type': 'BASED_ON'},
    {'source_text': 'GPT', 'target_text': 'Transformer', 'relation_type': 'BASED_ON'},
    {'source_text': 'Geoffrey Hinton', 'target_text': 'Deep Learning', 'relation_type': 'DEVELOPED'},
    {'source_text': 'Yann LeCun', 'target_text': 'Computer Vision', 'relation_type': 'DEVELOPED'},
]

graph_builder.build_from_extraction(sample_entities, sample_relations)
G = graph_builder.graph

print(f'Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## 2. Graph Visualization

In [ ]:
def visualize_graph(G, title='Knowledge Graph', figsize=(12, 8)):
    """Visualize the knowledge graph."""
    plt.figure(figsize=figsize)
    
    # Create layout
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    # Get node types for coloring
    node_types = [G.nodes[n].get('node_type', 'Unknown') for n in G.nodes()]
    unique_types = list(set(node_types))
    colors = plt.cm.Set3(np.linspace(0, 1, len(unique_types)))
    type_color_map = dict(zip(unique_types, colors))
    node_colors = [type_color_map[t] for t in node_types]
    
    # Draw graph
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1000, alpha=0.8)
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=15, alpha=0.6)
    nx.draw_networkx_labels(G, pos, font_size=8)
    
    # Add edge labels
    edge_labels = nx.get_edge_attributes(G, 'relation_type')
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=6)
    
    # Add legend
    legend_handles = [plt.scatter([], [], c=[type_color_map[t]], s=100, label=t) for t in unique_types]
    plt.legend(handles=legend_handles, loc='upper left')
    
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

visualize_graph(G)

## 3. Centrality Analysis

In [ ]:
def analyze_centrality(G):
    """Compute various centrality metrics."""
    metrics = {
        'degree': dict(G.degree()),
        'betweenness': nx.betweenness_centrality(G),
        'closeness': nx.closeness_centrality(G),
        'pagerank': nx.pagerank(G),
    }
    
    # Create DataFrame
    df = pd.DataFrame(metrics)
    df.index.name = 'node'
    
    return df

centrality_df = analyze_centrality(G)
print('Centrality Metrics:')
centrality_df.round(3)

In [ ]:
# Visualize centrality metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, metric in zip(axes.flat, ['degree', 'betweenness', 'closeness', 'pagerank']):
    sorted_df = centrality_df.sort_values(metric, ascending=True)
    ax.barh(range(len(sorted_df)), sorted_df[metric])
    ax.set_yticks(range(len(sorted_df)))
    ax.set_yticklabels(sorted_df.index, fontsize=8)
    ax.set_xlabel(metric.capitalize())
    ax.set_title(f'{metric.capitalize()} Centrality')

plt.tight_layout()
plt.show()

## 4. Community Detection

In [ ]:
# Initialize community detector
detector = CommunityDetector()

# Detect communities
communities = detector.detect_louvain(G)

print(f'Detected {len(communities)} communities:')
for i, community in enumerate(communities):
    print(f'  Community {i+1}: {list(community)}')

In [ ]:
def visualize_communities(G, communities):
    """Visualize graph with community coloring."""
    plt.figure(figsize=(12, 8))
    
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    # Create node-to-community mapping
    node_to_community = {}
    for i, community in enumerate(communities):
        for node in community:
            node_to_community[node] = i
    
    # Assign colors by community
    colors = plt.cm.Set1(np.linspace(0, 1, len(communities)))
    node_colors = [colors[node_to_community.get(n, 0)] for n in G.nodes()]
    
    nx.draw(G, pos, node_color=node_colors, node_size=1000, 
            with_labels=True, font_size=8, arrows=True, arrowsize=15)
    
    plt.title('Communities in Knowledge Graph')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

visualize_communities(G, communities)

## 5. Graph Statistics Summary

In [ ]:
def graph_statistics(G):
    """Compute comprehensive graph statistics."""
    stats = {
        'nodes': G.number_of_nodes(),
        'edges': G.number_of_edges(),
        'density': nx.density(G),
        'avg_degree': np.mean([d for n, d in G.degree()]),
        'is_connected': nx.is_weakly_connected(G) if G.is_directed() else nx.is_connected(G),
    }
    
    # Additional metrics for connected graphs
    if G.is_directed():
        if nx.is_weakly_connected(G):
            undirected = G.to_undirected()
            stats['diameter'] = nx.diameter(undirected)
            stats['avg_path_length'] = nx.average_shortest_path_length(undirected)
    else:
        if nx.is_connected(G):
            stats['diameter'] = nx.diameter(G)
            stats['avg_path_length'] = nx.average_shortest_path_length(G)
    
    stats['clustering_coefficient'] = nx.average_clustering(G.to_undirected())
    
    return stats

stats = graph_statistics(G)
print('Graph Statistics:')
for key, value in stats.items():
    if isinstance(value, float):
        print(f'  {key}: {value:.4f}')
    else:
        print(f'  {key}: {value}')